# 00 · Generación de datos faltantes (MCAR)

**MCDI501 - Estadística Computacional para la Toma de Decisiones** · Grupo 8

El conjunto *Predict Students' Dropout and Academic Success* viene completo (sin datos faltantes). El apunte
*Bases de datos disponibles para el proyecto del curso* (sección 2) exige que, en ese caso, el equipo genere
faltantes de forma artificial **antes de comenzar el análisis**, con estas reglas:

- una o dos variables numéricas relevantes;
- entre 10 % y 15 % de sus valores eliminados de forma completamente aleatoria (MCAR);
- semilla fija para que el proceso sea reproducible;
- guardar los valores originales para medir después el error de imputación;
- ejecutar la rutina **una sola vez** y trabajar todo el proyecto con la versión "con faltantes".

Variables elegidas: `Admission grade` (nota de admisión) y `Age at enrollment` (edad al ingresar), con 12 % de
faltantes cada una. Ambas son numéricas y relevantes para explicar la deserción.

In [ ]:
import os
import numpy as np
import pandas as pd

SEMILLA = 42
PROPORCION = 0.12
VARIABLES = ['Admission grade', 'Age at enrollment']

RUTA_RAW = '../data/raw/data.csv'
RUTA_FALTANTES = '../data/processed/data_con_faltantes.csv'
RUTA_ORIGINALES = '../data/processed/valores_originales.csv'

df = pd.read_csv(RUTA_RAW, sep=';')
df.columns = df.columns.str.strip()          # una columna trae un tabulador al final del nombre
print('Dimensiones:', df.shape)
print('Datos faltantes en el archivo original:', df.isna().sum().sum())

In [ ]:
# La rutina se ejecuta una sola vez: si el archivo ya existe, no se vuelve a generar
if os.path.exists(RUTA_FALTANTES):
    print('El archivo con faltantes ya existe; no se genera de nuevo.')
else:
    rng = np.random.default_rng(SEMILLA)
    df_f = df.copy()
    df_f.insert(0, 'id_estudiante', np.arange(1, len(df) + 1))   # identificador para trazar cada fila
    originales = []
    for var in VARIABLES:
        n_eliminar = int(round(PROPORCION * len(df_f)))
        filas = rng.choice(len(df_f), size=n_eliminar, replace=False)   # cada fila tiene la misma probabilidad
        for f in filas:
            originales.append({'id_estudiante': df_f.at[f, 'id_estudiante'], 'variable': var,
                               'valor_original': df_f.at[f, var]})
        df_f.loc[filas, var] = np.nan

    os.makedirs('../data/processed', exist_ok=True)
    df_f.to_csv(RUTA_FALTANTES, index=False)
    pd.DataFrame(originales).to_csv(RUTA_ORIGINALES, index=False)
    print('Archivos generados:', RUTA_FALTANTES, 'y', RUTA_ORIGINALES)

In [ ]:
# Verificación
df_f = pd.read_csv(RUTA_FALTANTES)
resumen = pd.DataFrame({'Faltantes': df_f[VARIABLES].isna().sum(),
                        '%': (df_f[VARIABLES].isna().mean() * 100).round(2)})
print(resumen, '\n')
print('Filas con al menos un faltante:', df_f[VARIABLES].isna().any(axis=1).sum())
print('Valores originales guardados  :', len(pd.read_csv(RUTA_ORIGINALES)))

**Comprobación MCAR.** Como los valores se eliminaron al azar, la distribución de la variable objetivo debe ser
casi la misma entre las filas con y sin faltante.

In [ ]:
for var in VARIABLES:
    tabla = pd.crosstab(df_f[var].isna(), df_f['Target'], normalize='index').round(3) * 100
    tabla.index = ['Con valor', 'Faltante']
    print(var); print(tabla, '\n')